# Spotify

In [2]:
import requests

url = "spotify.com/in-en/premium/"
api = "http://web.archive.org/cdx/search/cdx"
params = {
    "url": url,
    "output": "json",
    "fl": "timestamp,original",
    "filter": "statuscode:200",
    "collapse": "timestamp:6",   # keep one snapshot per month
}

rows = requests.get(api, params=params).json()[1:]  # first row is the header
print(len(rows), "snapshots found")

for ts, original in rows[:5]:
    print(f"https://web.archive.org/web/{ts}/{original}")

65 snapshots found
https://web.archive.org/web/20210224120829/https://www.spotify.com/in-en/premium/
https://web.archive.org/web/20210414202412/https://www.spotify.com/in-en/premium/
https://web.archive.org/web/20210519000054/https://www.spotify.com/in-en/premium/
https://web.archive.org/web/20210615164021/https://www.spotify.com/in-en/premium/
https://web.archive.org/web/20210717064921/https://www.spotify.com/in-en/premium/


In [3]:
import re, time
from datetime import datetime

def get_prices(ts, original):
    link = f"https://web.archive.org/web/{ts}id_/{original}"
    r = requests.get(link, timeout=30)
    r.encoding = "utf-8"
    return re.findall(r"(?:₹|&#8377;|Rs\.?)\s?(\d[\d,]*)", r.text)

for ts, original in rows[:5]:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    print(date, get_prices(ts, original))
    time.sleep(1)

2021-02-24 ['7', '25', '179', '7', '119', '149', '179', '7', '25', '179', '7', '119', '149', '179']
2021-04-14 ['199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199']
2021-05-19 ['199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199']
2021-06-15 ['199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199']
2021-07-17 ['199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199', '199', '7', '25', '7', '129', '165', '199']


In [4]:
import pandas as pd

def clean_prices(raw):
    nums = {int(p.replace(",", "")) for p in raw}   # set removes repeats
    nums = [n for n in nums if n >= 100]            # drops 7 and 25
    return sorted(nums)                             # lowest to highest

records = []
for ts, original in rows[:5]:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    nums = clean_prices(get_prices(ts, original))

    if len(nums) == 3:
        individual, duo, family = nums
        records.append({
            "service": "Spotify", "region": "IN", "date": date,
            "individual": individual, "duo": duo, "family": family,
            "source": "wayback",
        })
    else:
        print("check manually:", date, nums)

    time.sleep(1)

df = pd.DataFrame(records)
print(df)

   service region        date  individual  duo  family   source
0  Spotify     IN  2021-02-24         119  149     179  wayback
1  Spotify     IN  2021-04-14         129  165     199  wayback
2  Spotify     IN  2021-05-19         129  165     199  wayback
3  Spotify     IN  2021-06-15         129  165     199  wayback
4  Spotify     IN  2021-07-17         129  165     199  wayback


In [12]:
records = []
problems = []

for ts, original in rows:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    try:
        nums = clean_prices(get_prices(ts, original))
        if len(nums) == 3:
            individual, duo, family = nums
            records.append({
                "service": "Spotify", "region": "IN", "date": date,
                "individual": individual, "duo": duo, "family": family,
                "source": "wayback",
            })
        else:
            problems.append((date, nums))
    except Exception as e:
        problems.append((date, str(e)))
    time.sleep(1)

df = pd.DataFrame(records)
df.to_csv("spotify_in.csv", index=False)

print(len(df), "clean rows,", len(problems), "problems")
for p in problems:
    print(p)

22 clean rows, 43 problems
(datetime.date(2021, 10, 7), [165, 199, 250, 939])
(datetime.date(2022, 3, 7), [119, 149, 179, 799])
(datetime.date(2022, 9, 2), [119, 179])
(datetime.date(2022, 11, 5), 'HTTPSConnectionPool(host=\'web.archive.org\', port=443): Max retries exceeded with url: /web/20221105223637id_/https://www.spotify.com/in-en/premium/ (Caused by NewConnectionError("HTTPSConnection(host=\'web.archive.org\', port=443): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))')
(datetime.date(2022, 12, 2), 'HTTPSConnectionPool(host=\'web.archive.org\', port=443): Max retries exceeded with url: /web/20221202211755id_/https://www.spotify.com/in-en/premium/ (Caused by NewConnectionError("HTTPSConnection(host=\'web.archive.org\', port=443): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))')
(datetime.date(2023, 1, 5), 'HTTPS

In [14]:
failed_dates = {d for d, info in problems if isinstance(info, str)}

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (student project, price research)"})

def get_prices_retry(ts, original, tries=4):
    link = f"https://web.archive.org/web/{ts}id_/{original}"
    for attempt in range(tries):
        try:
            r = session.get(link, timeout=30)
            r.raise_for_status()
            r.encoding = "utf-8"
            return re.findall(r"(?:₹|&#8377;|Rs\.?)\s?(\d[\d,]*)", r.text)
        except requests.exceptions.RequestException:
            time.sleep(5 * (attempt + 1))
    raise RuntimeError("gave up after retries")

new_problems = []
for ts, original in rows:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    if date not in failed_dates:
        continue
    try:
        nums = clean_prices(get_prices_retry(ts, original))
        if len(nums) == 3:
            individual, duo, family = nums
            records.append({
                "service": "Spotify", "region": "IN", "date": date,
                "individual": individual, "duo": duo, "family": family,
                "source": "wayback",
            })
        else:
            new_problems.append((date, nums))
    except Exception as e:
        new_problems.append((date, str(e)))
    time.sleep(3)

df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
df.to_csv("spotify_in.csv", index=False)

print(len(df), "clean rows now")
print(len(new_problems), "still unresolved")
for p in new_problems:
    print(p)

35 clean rows now
9 still unresolved
(datetime.date(2025, 9, 23), [115, 138, 139, 149, 179, 189, 229, 249, 278, 419, 539, 829, 1079, 1390, 2149])
(datetime.date(2025, 10, 8), [115, 138, 139, 149, 179, 189, 229, 249, 278, 419, 539, 829, 1079, 1390, 2149])
(datetime.date(2025, 11, 6), [])
(datetime.date(2025, 12, 2), [139, 166, 199, 299, 389, 1999])
(datetime.date(2026, 1, 1), [139, 166, 199, 299, 389, 1999])
(datetime.date(2026, 2, 3), [139, 166, 199, 299, 389, 1999])
(datetime.date(2026, 3, 3), [139, 166, 199, 299, 389, 1999])
(datetime.date(2026, 4, 17), [139, 166, 199, 299, 389, 1999])
(datetime.date(2026, 6, 16), [115, 139, 278, 299, 1390])


In [16]:
def get_html(ts, original, tries=4):
    link = f"https://web.archive.org/web/{ts}id_/{original}"
    for attempt in range(tries):
        try:
            r = session.get(link, timeout=30)
            r.raise_for_status()
            r.encoding = "utf-8"
            return r.text
        except requests.exceptions.RequestException:
            time.sleep(5 * (attempt + 1))
    return ""

target = "20250226"
ts, original = next(r for r in rows if r[0].startswith(target))
html = get_html(ts, original)

text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.S)
text = re.sub(r"<[^>]+>", " ", text)
text = re.sub(r"\s+", " ", text)

for name in ["Individual", "Duo", "Family", "Student"]:
    for m in list(re.finditer(name, text))[:3]:
        print(name, "->", text[m.start(): m.start() + 120])
        print()

Individual -> Individual Skip to content Spotify Premium Premium Individual 1 account - For one person. Premium Duo 2 accounts - For c

Individual -> Individual 1 account - For one person. Premium Duo 2 accounts - For couples under one roof. Premium Family 6 accounts - 

Individual -> Individual View all plans Premium Individual only. ₹119 for 4 months, then ₹119 per month after. Offer only available if

Duo -> Duo 2 accounts - For couples under one roof. Premium Family 6 accounts - For family members under one roof. Premium Stud

Duo -> Duo ₹149 for 2 months ₹149/month after 2 Premium accounts Cancel anytime Subscribe or one-time payment Get Premium Duo P

Duo -> Duo Premium Duo Pay once in advance, top up when you want. Does not auto-renew. ₹1,799 for 1 year Equivalent to ₹149.92 

Family -> Family 6 accounts - For family members under one roof. Premium Student 1 account - Discount for eligible students. Suppo

Family -> Family ₹179 for 2 months ₹179/month after Up to 6 Premium accou

In [18]:
PLANS = ["Individual", "Duo", "Family", "Student"]
OTHER = "Individual|Duo|Family|Student"

def page_text(html):
    text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text)

def extract_plans(text):
    out = {}
    for name in PLANS:
        pattern = (name + r"(?:(?!" + OTHER + r").){0,120}?"
                   r"₹\s?([\d,]+)\s*(?:/|per )\s*month after")
        m = re.search(pattern, text)
        out[name.lower()] = int(m.group(1).replace(",", "")) if m else None
    return out

done = set(df["date"])
todo = [(ts, o) for ts, o in rows
        if datetime.strptime(ts, "%Y%m%d%H%M%S").date() not in done]

new_records, still_bad = [], []
for ts, original in todo:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    html = get_html(ts, original)
    plans = extract_plans(page_text(html)) if html else {}
    if plans.get("individual") and plans.get("duo") and plans.get("family"):
        new_records.append({"service": "Spotify", "region": "IN", "date": date,
                            **plans, "source": "wayback"})
    else:
        still_bad.append((date, plans))
    time.sleep(3)

df = pd.concat([df, pd.DataFrame(new_records)]).sort_values("date").reset_index(drop=True)
df.to_csv("spotify_in.csv", index=False)

print(len(df), "clean rows now")
print(len(still_bad), "still unresolved")
for b in still_bad:
    print(b)

51 clean rows now
14 still unresolved
(datetime.date(2021, 10, 7), {'individual': None, 'duo': None, 'family': None, 'student': None})
(datetime.date(2022, 9, 2), {'individual': 119, 'duo': None, 'family': 179, 'student': 59})
(datetime.date(2023, 12, 4), {'individual': 119, 'duo': None, 'family': None, 'student': 59})
(datetime.date(2025, 11, 6), {'individual': None, 'duo': None, 'family': None, 'student': None})
(datetime.date(2025, 12, 2), {'individual': None, 'duo': None, 'family': None, 'student': 99})
(datetime.date(2026, 1, 1), {'individual': None, 'duo': None, 'family': None, 'student': 99})
(datetime.date(2026, 2, 3), {'individual': None, 'duo': None, 'family': None, 'student': 99})
(datetime.date(2026, 3, 3), {'individual': None, 'duo': None, 'family': None, 'student': 99})
(datetime.date(2026, 4, 17), {'individual': None, 'duo': None, 'family': None, 'student': 99})
(datetime.date(2026, 5, 6), {'individual': None, 'duo': None, 'family': None, 'student': 99})
(datetime.date(2

In [19]:
target = "20260303"
ts, original = next(r for r in rows if r[0].startswith(target))
text = page_text(get_html(ts, original))

for m in list(re.finditer(r"₹", text))[:12]:
    print(text[max(0, m.start() - 70): m.start() + 80])
    print()

 Spotify Premium (India) - Try 3 months for ₹99 Skip to content Spotify Premium plans Premium Lite All the Premium essential

r eligible students. Support Download Sign up Log in Try 3 months for ₹99 Try 3 months for ₹99 Menu The ultimate home for music artists fans live even

Support Download Sign up Log in Try 3 months for ₹99 Try 3 months for ₹99 Menu The ultimate home for music artists fans live events videos Try 3 month

imate home for music artists fans live events videos Try 3 months for ₹99 View all plans Premium Standard only. ₹99 for 3 months, then ₹199 per month 

nts videos Try 3 months for ₹99 View all plans Premium Standard only. ₹99 for 3 months, then ₹199 per month after. Offer only available if you haven’t

 for ₹99 View all plans Premium Standard only. ₹99 for 3 months, then ₹199 per month after. Offer only available if you haven’t tried Premium before. 

albums in order, and skip as many times as you want. Try 3 months for ₹99 View all plans Premium Standard only. ₹9

In [20]:
from collections import Counter

names = Counter(re.findall(r"Premium (\w+)", text))
print(names)
print()

for name in names:
    if name in ("Lite", "Standard", "only", "plans"):
        continue
    for m in list(re.finditer("Premium " + name, text))[:2]:
        print(text[m.start(): m.start() + 160])
        print()

Counter({'Standard': 7, 'Lite': 4, 'Platinum': 4, 'Student': 3, 'before': 3, 'plans': 1, 'essentials': 1, 'Watch': 1, 'Listen': 1})

Premium essentials - 1 account. Premium Standard Your music. Your way. Everywhere - 1 account. Premium Platinum Listen on a whole new level - Up to 3 accounts. 

Premium Platinum Listen on a whole new level - Up to 3 accounts. Premium Student 1 account - Discount for eligible students. Support Download Sign up Log in Try

Premium Platinum ₹299 / month Up to 3 Platinum accounts Download to listen offline Lossless audio quality (up to ~24-bit/44.1kHz) Mix your playlists Your person

Premium Student 1 account - Discount for eligible students. Support Download Sign up Log in Try 3 months for ₹99 Try 3 months for ₹99 Menu The ultimate home for

Premium Student ₹99 for 2 months ₹99/month after 1 verified Standard account Download to listen offline Very high audio quality (up to ~320kbps) Cancel anytime 

Premium before. Terms apply. Offer ends 13 March 2026. As

In [21]:
import html as htmllib

def extract_new(text):
    text = htmllib.unescape(text)
    out = {}
    for name in ["Lite", "Platinum"]:
        m = re.search(r"Premium " + name + r"\s*₹\s?([\d,]+)\s*/\s*month", text)
        out[name] = int(m.group(1).replace(",", "")) if m else None
    for name in ["Standard", "Student"]:
        m = re.search(
            r"Premium " + name + r"\s*[^₹]{0,40}₹[\d,]+ for \d+ months?,?\s*(?:then )?"
            r"₹\s?([\d,]+)\s*(?:/|per )\s*month", text)
        out[name] = int(m.group(1).replace(",", "")) if m else None
    return out

new_rows, bad2 = [], []
for ts, original in rows:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    if date < datetime(2025, 12, 2).date():
        continue
    prices = extract_new(page_text(get_html(ts, original)))
    if any(prices.values()):
        for plan, price in prices.items():
            if price:
                new_rows.append({"service": "Spotify", "region": "IN", "date": date,
                                 "plan": plan, "regular_price": price,
                                 "source": "wayback"})
    else:
        bad2.append(date)
    time.sleep(3)

df_new = pd.DataFrame(new_rows)
print(df_new.pivot(index="date", columns="plan", values="regular_price"))
print("no prices found:", bad2)

plan         Lite  Platinum  Standard  Student
date                                          
2025-12-02  139.0     299.0     199.0     99.0
2026-01-01  139.0     299.0     199.0     99.0
2026-02-03  139.0     299.0     199.0     99.0
2026-03-03  139.0     299.0     199.0     99.0
2026-04-17  139.0     299.0     199.0     99.0
2026-05-06  139.0     299.0     199.0     99.0
2026-06-16    NaN     299.0     139.0     69.0
2026-07-15    NaN     299.0     139.0     69.0
2026-08-15    NaN     299.0       NaN     69.0
2026-09-07    NaN     299.0     139.0     69.0
no prices found: []


In [22]:
for target in ["20260616", "20260815"]:
    ts, original = next(r for r in rows if r[0].startswith(target))
    text = htmllib.unescape(page_text(get_html(ts, original)))
    print("=====", target)
    for name in ["Lite", "Standard", "Platinum", "Student"]:
        for m in list(re.finditer("Premium " + name, text))[:3]:
            print(name, "->", text[m.start(): m.start() + 140])
            print()
    time.sleep(3)

===== 20260616
Standard -> Premium Standard Your music. Your way. Everywhere - 1 account. Premium Platinum Listen on a whole new level - Up to 3 accounts. Premium Stud

Standard -> Premium Standard only. ₹99 for 3 months, then ₹139 per month after. Offer only available if you haven’t tried Premium before. Terms apply. O

Standard -> Premium Standard Made for fans who want more Ad-free music listening All the music, without all the ads. Just nonstop beats. Very high audio

Platinum -> Premium Platinum Listen on a whole new level - Up to 3 accounts. Premium Student 1 account - Discount for eligible students. Support Downloa

Platinum -> Premium Platinum ₹299 / month Up to 3 Platinum accounts Download to listen offline Lossless audio quality (up to ~24-bit/44.1kHz) Mix your p

Platinum -> Premium Platinum For up to 3 individuals residing at the same address. Terms apply. Savings available Premium Student ₹69 for 2 months ₹69/m

Student -> Premium Student 1 account - Discount for eligible 

In [23]:
def extract_new(text):
    text = htmllib.unescape(text)
    out = {}
    for name in ["Lite", "Platinum"]:
        m = re.search(r"Premium " + name + r"\s*₹\s?([\d,]+)\s*/\s*month", text)
        out[name] = int(m.group(1).replace(",", "")) if m else None
    for name in ["Standard", "Student"]:
        m = re.search(
            r"Premium " + name + r"\s*[^₹]{0,40}₹[\d,]+ for \d+ (?:months?|years?),?\s*(?:then )?"
            r"₹\s?([\d,]+)\s*(?:/|per )\s*month", text)
        out[name] = int(m.group(1).replace(",", "")) if m else None
    return out

date = datetime(2026, 8, 15).date()
ts, original = next(r for r in rows if r[0].startswith("20260815"))
prices = extract_new(page_text(get_html(ts, original)))
print(prices)

df_new = df_new[df_new["date"] != date]
fix = [{"service": "Spotify", "region": "IN", "date": date, "plan": p,
        "regular_price": v, "source": "wayback"}
       for p, v in prices.items() if v]
df_new = pd.concat([df_new, pd.DataFrame(fix)]).sort_values(["date", "plan"])

df_new.to_csv("spotify_in_new_plans.csv", index=False)
print(df_new.pivot(index="date", columns="plan", values="regular_price"))

{'Lite': None, 'Platinum': 299, 'Standard': 139, 'Student': 69}
plan         Lite  Platinum  Standard  Student
date                                          
2025-12-02  139.0     299.0     199.0     99.0
2026-01-01  139.0     299.0     199.0     99.0
2026-02-03  139.0     299.0     199.0     99.0
2026-03-03  139.0     299.0     199.0     99.0
2026-04-17  139.0     299.0     199.0     99.0
2026-05-06  139.0     299.0     199.0     99.0
2026-06-16    NaN     299.0     139.0     69.0
2026-07-15    NaN     299.0     139.0     69.0
2026-08-15    NaN     299.0     139.0     69.0
2026-09-07    NaN     299.0     139.0     69.0


In [24]:
old_long = df.melt(
    id_vars=["service", "region", "date", "source"],
    value_vars=["individual", "duo", "family", "student"],
    var_name="plan", value_name="regular_price",
).dropna(subset=["regular_price"])
old_long["plan"] = old_long["plan"].str.capitalize()

spotify = pd.concat([old_long, df_new], ignore_index=True)
spotify["date"] = pd.to_datetime(spotify["date"])
spotify["regular_price"] = spotify["regular_price"].astype(int)
spotify = spotify.sort_values(["plan", "date"]).reset_index(drop=True)
spotify.to_csv("spotify_in_long.csv", index=False)

print(len(spotify), "rows")
print(spotify.groupby("plan")["date"].agg(["min", "max", "count"]))

spotify["prev_price"] = spotify.groupby("plan")["regular_price"].shift()
changes = spotify[spotify["prev_price"].notna()
                  & (spotify["regular_price"] != spotify["prev_price"])].copy()
changes["change_pct"] = ((changes["regular_price"] / changes["prev_price"] - 1) * 100).round(1)
print()
print(changes[["date", "plan", "prev_price", "regular_price", "change_pct"]].to_string(index=False))

204 rows
                  min        max  count
plan                                   
Duo        2021-02-24 2025-10-08     51
Family     2021-02-24 2025-10-08     51
Individual 2021-02-24 2025-10-08     51
Lite       2025-12-02 2026-05-06      6
Platinum   2025-12-02 2026-09-07     10
Standard   2025-12-02 2026-09-07     10
Student    2024-06-21 2026-09-07     25

      date       plan  prev_price  regular_price  change_pct
2021-04-14        Duo       149.0            165        10.7
2021-08-10        Duo       165.0            199        20.6
2021-09-21        Duo       199.0            165       -17.1
2022-02-18        Duo       165.0            149        -9.7
2025-08-04        Duo       149.0            179        20.1
2021-04-14     Family       179.0            199        11.2
2021-08-10     Family       199.0            719       261.3
2021-09-21     Family       719.0            199       -72.3
2022-02-18     Family       199.0            179       -10.1
2025-08-04     Famil

In [26]:
sp = pd.read_csv("spotify_in_long.csv", parse_dates=["date"])
sp["flag"] = ""

sp.loc[sp["date"] == "2021-08-10", "flag"] = "bad_extraction"
sp.loc[(sp["date"] == "2021-12-01") & (sp["plan"] == "Individual"), "flag"] = "suspect"

sp.to_csv("spotify_in_flagged.csv", index=False)

clean = sp[sp["flag"] != "bad_extraction"].copy()
clean.to_csv("spotify_in_clean.csv", index=False)
print(len(sp), "rows flagged file,", len(clean), "rows clean file")

204 rows flagged file, 201 rows clean file


In [27]:
sp = pd.read_csv("spotify_in_clean.csv", parse_dates=["date"])

sp["plan_group"] = sp["plan"].replace({
    "Individual": "Individual/Lite",
    "Lite": "Individual/Lite",
})

sp.to_csv("spotify_in_clean.csv", index=False)

print(sp.groupby("plan_group")["date"].agg(["min", "max", "count"]))

                       min        max  count
plan_group                                  
Duo             2021-02-24 2025-10-08     50
Family          2021-02-24 2025-10-08     50
Individual/Lite 2021-02-24 2026-05-06     56
Platinum        2025-12-02 2026-09-07     10
Standard        2025-12-02 2026-09-07     10
Student         2024-06-21 2026-09-07     25


# Youtube 

In [32]:
events = pd.DataFrame([
    ("Individual", "2021-02-01", 129),
    ("Individual", "2024-09-01", 149),
    ("Family",     "2021-02-01", 189),
    ("Family",     "2024-09-01", 299),
    ("Student",    "2021-02-01", 79),
    ("Student",    "2024-09-01", 89),
    ("Lite",       "2025-10-01", 89),
], columns=["plan", "start", "price"])
events["start"] = pd.to_datetime(events["start"])

months = pd.date_range("2021-02-01", "2026-09-01", freq="MS")
rows = []
for plan, g in events.groupby("plan"):
    g = g.sort_values("start")
    for m in months:
        active = g[g["start"] <= m]
        if len(active):
            rows.append({"service": "YouTube", "region": "IN", "date": m,
                         "plan": plan, "regular_price": int(active.iloc[-1]["price"]),
                         "source": "news_reports"})

yt_in = pd.DataFrame(rows).sort_values(["plan", "date"])
yt_in.to_csv("youtube_in.csv", index=False)

print(len(yt_in), "rows")
print(yt_in.groupby("plan")["regular_price"].agg(["min", "max", "count"]))

216 rows
            min  max  count
plan                       
Family      189  299     68
Individual  129  149     68
Lite         89   89     12
Student      79   89     68


In [33]:
print(yt_in.head(10))

   service region       date    plan  regular_price        source
0  YouTube     IN 2021-02-01  Family            189  news_reports
1  YouTube     IN 2021-03-01  Family            189  news_reports
2  YouTube     IN 2021-04-01  Family            189  news_reports
3  YouTube     IN 2021-05-01  Family            189  news_reports
4  YouTube     IN 2021-06-01  Family            189  news_reports
5  YouTube     IN 2021-07-01  Family            189  news_reports
6  YouTube     IN 2021-08-01  Family            189  news_reports
7  YouTube     IN 2021-09-01  Family            189  news_reports
8  YouTube     IN 2021-10-01  Family            189  news_reports
9  YouTube     IN 2021-11-01  Family            189  news_reports


# JioHotstar

In [35]:
events = pd.DataFrame([
    # plan, billing, start, price
    ("Mobile",  "quarterly", "2025-02-01", 149),
    ("Mobile",  "annual",    "2025-02-01", 499),
    ("Super",   "quarterly", "2025-02-01", 299),
    ("Super",   "annual",    "2025-02-01", 899),
    ("Premium", "quarterly", "2025-02-01", 499),
    ("Premium", "annual",    "2025-02-01", 1499),
    ("Super",   "quarterly", "2026-02-01", 349),
    ("Super",   "annual",    "2026-02-01", 1099),
    ("Premium", "quarterly", "2026-02-01", 699),
    ("Premium", "annual",    "2026-02-01", 2199),
    ("Mobile",  "monthly",   "2026-02-01", 79),
    ("Super",   "monthly",   "2026-02-01", 149),
    ("Premium", "monthly",   "2026-02-01", 299),
], columns=["plan", "billing", "start", "price"])
events["start"] = pd.to_datetime(events["start"])

months = pd.date_range("2025-02-01", "2026-09-01", freq="MS")
rows = []
for (plan, billing), g in events.groupby(["plan", "billing"]):
    g = g.sort_values("start")
    for m in months:
        active = g[g["start"] <= m]
        if len(active):
            rows.append({"service": "JioHotstar", "region": "IN", "date": m,
                         "plan": plan, "billing": billing,
                         "regular_price": int(active.iloc[-1]["price"]),
                         "source": "news_reports"})

hs = pd.DataFrame(rows).sort_values(["plan", "billing", "date"])
hs.to_csv("jiohotstar_in.csv", index=False)

print(len(hs), "rows")
print(hs.groupby(["plan", "billing"])["regular_price"].agg(["min", "max", "count"]))

144 rows
                    min   max  count
plan    billing                     
Mobile  annual      499   499     20
        monthly      79    79      8
        quarterly   149   149     20
Premium annual     1499  2199     20
        monthly     299   299      8
        quarterly   499   699     20
Super   annual      899  1099     20
        monthly     149   149      8
        quarterly   299   349     20


# Zee5

In [37]:
def discover(site, limit=30):
    params = {
        "url": site,
        "matchType": "prefix",
        "output": "json",
        "fl": "original",
        "filter": ["statuscode:200", "original:.*(subscri|premium|plan|pack|pricing).*"],
        "collapse": "urlkey",
        "limit": limit,
    }
    for attempt in range(4):
        try:
            r = session.get("https://web.archive.org/cdx/search/cdx",
                            params=params, timeout=60)
            if r.status_code == 200 and r.text.strip():
                return [row[0] for row in r.json()[1:]]
            print("  got status", r.status_code, r.text[:80])
        except Exception as e:
            print("  error:", str(e)[:80])
        time.sleep(10 * (attempt + 1))
    return []

for site in ["sonyliv.com/", "zee5.com/"]:
    urls = discover(site)
    print(site, "->", len(urls), "pages found")
    for u in urls:
        print("  ", u)
    time.sleep(5)

sonyliv.com/ -> 30 pages found
   http://www.sonyliv.com:80/already-subscribed
   https://www.sonyliv.com/assets/images/premium_icon_01.png
   http://www.sonyliv.com:80/assets/listing/premium-offers-band-1
   https://www.sonyliv.com/cancelsubscription
   https://www.sonyliv.com/compareplans
   https://www.sonyliv.com/custompage/premium-2264
   https://www.sonyliv.com/custompage/premium-2264?utm_source=zoutons&utm_medium=cps&utm_campaogn=zout01
   https://www.sonyliv.com/custompage/premium-23503
   https://www.sonyliv.com/custompage/premium-35221
   https://www.sonyliv.com/custompage/premium-35223
   https://www.sonyliv.com/custompage/premium-400
   http://www.sonyliv.com:80/details/clips/5139580684001/Magnificent-Ditya-Bhande-from-Mumbai-with-her-power-packed-urban-hip-hop-performance---Super-Dancer
   http://www.sonyliv.com:80/details/clips/5302408966001/Federer-has-no-plans-to-stop-playing
   https://www.sonyliv.com/details/clips/5668215234001/Eggplant-Crostini---Cook-Smart---Season-

In [38]:
for u in ["sonyliv.com/compareplans",
          "sonyliv.com/custompage/premium-2264",
          "sonyliv.com/custompage/premium-400",
          "zee5.com/myaccount/subscription"]:
    probe2(u)

sonyliv.com/compareplans -> 1 yearly snapshots
20211103 | size: 133786 | rupee prices: 0 []
sonyliv.com/custompage/premium-2264 -> 5 yearly snapshots
20210729 | size: 2138670 | rupee prices: 0 []
20220123 | size: 1130866 | rupee prices: 0 []
20230524 | size: 1424687 | rupee prices: 0 []
20240622 | size: 1048566 | rupee prices: 0 []
20250718 | size: 1982029 | rupee prices: 3 ['₹299', '₹299', '₹299']
sonyliv.com/custompage/premium-400 -> 1 yearly snapshots
20200616 | size: 92761 | rupee prices: 1 ['Rs. 99']
zee5.com/myaccount/subscription -> 6 yearly snapshots
20191008 | size: 8820 | rupee prices: 0 []
20220914 | size: 405974 | rupee prices: 16 ['₹58', 'Rs 499', 'Rs. 50', '₹42', '₹42']
20230130 | size: 306071 | rupee prices: 14 ['Rs 499', '₹42', '₹42', 'Rs. 499', '₹ 499']
20240102 | size: 248902 | rupee prices: 12 ['₹649', '₹599', '₹999', '₹899', '₹1999']
20250116 | size: 220442 | rupee prices: 11 ['₹ 399', '₹ 299', '₹ 299', '₹ 199', '₹ 249']
20260103 | size: 220790 | rupee prices: 0 []


In [39]:
data = cdx_list("zee5.com/myaccount/subscription")
ts, original = next(r for r in data if r[0].startswith("2024"))
raw = htmllib.unescape(get_html(ts, original))

hits = list(re.finditer(r"(?:₹|(?i:\\u20b9)|Rs\.?)\s?\d[\d,]*", raw))
print(ts[:8], "->", len(hits), "price mentions")

for m in hits[:12]:
    print(raw[max(0, m.start() - 90): m.start() + 60].replace("\n", " "))
    print("---")

  error: HTTPSConnectionPool(host='web.archive.org', port=443): Read timed out. (read tim
20240102 -> 12 price mentions
:line-through;text-decoration:line-through;color:#E0E0E0;}</style><div class="css-o9x45v">₹649</div><style data-emotion="css rces6j">.css-rces6j{font-
---
.125rem;font-weight:400;font-weight:700;line-height:24px;}</style><div class="css-rces6j">₹599</div><style data-emotion="css 1ri5aq9">.css-1ri5aq9{fon
---
v><div class="css-dgik2l"><div class="css-c3i1sv">Premium HD</div><div class="css-o9x45v">₹999</div><div class="css-rces6j">₹899</div><div class="css-
---
ss="css-c3i1sv">Premium HD</div><div class="css-o9x45v">₹999</div><div class="css-rces6j">₹899</div><div class="css-1ri5aq9">12 months</div></div></di
---
v><div class="css-dgik2l"><div class="css-c3i1sv">Premium 4k</div><div class="css-o9x45v">₹1999</div><div class="css-rces6j">₹1199</div><div class="cs
---
s="css-c3i1sv">Premium 4k</div><div class="css-o9x45v">₹1999</div><div class="css-rces6j">₹1199</div><div

In [40]:
def cdx_monthly(url):
    params = {"url": url, "output": "json", "fl": "timestamp,original",
              "filter": "statuscode:200", "collapse": "timestamp:6"}
    for attempt in range(4):
        try:
            r = session.get("https://web.archive.org/cdx/search/cdx",
                            params=params, timeout=60)
            if r.status_code == 200 and r.text.strip():
                return r.json()[1:]
        except Exception as e:
            print("  error:", str(e)[:80])
        time.sleep(10 * (attempt + 1))
    return []

def zee_cards(html):
    t = htmllib.unescape(html)
    t = re.sub(r"<style.*?</style>|<script.*?</script>", " ", t, flags=re.S)
    t = re.sub(r"<[^>]+>", "|", t)
    t = re.sub(r"\|[\s|]*\|", "|", t)
    pat = r"([^|]{2,40})\|₹\s?([\d,]+)(?:\|₹\s?([\d,]+))?\|(\d+ (?:months?|days?|years?))"
    out = []
    for name, p1, p2, dur in re.findall(pat, t):
        price = int((p2 or p1).replace(",", ""))
        list_price = int(p1.replace(",", "")) if p2 else price
        out.append((name.strip(), dur, list_price, price))
    return out

snaps = cdx_monthly("zee5.com/myaccount/subscription")
print(len(snaps), "monthly snapshots")

zrows, zbad = [], []
for ts, original in snaps:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    html = get_html(ts, original)
    cards = zee_cards(html) if html else []
    if cards:
        for name, dur, lp, p in cards:
            zrows.append({"service": "ZEE5", "region": "IN", "date": date,
                          "plan": name, "duration": dur, "list_price": lp,
                          "price": p, "source": "wayback"})
    else:
        zbad.append(date)
    time.sleep(3)

zee = pd.DataFrame(zrows)
zee.to_csv("zee5_in_raw.csv", index=False)
print(len(zee), "plan rows from", zee["date"].nunique(), "snapshots")
print(len(zbad), "snapshots with no cards:", zbad)
print(zee.groupby(["plan", "duration"])["price"].agg(["min", "max", "count"]))

35 monthly snapshots
29 plan rows from 10 snapshots
25 snapshots with no cards: [datetime.date(2019, 10, 8), datetime.date(2022, 9, 14), datetime.date(2022, 10, 4), datetime.date(2022, 11, 11), datetime.date(2023, 1, 30), datetime.date(2023, 3, 6), datetime.date(2023, 4, 1), datetime.date(2023, 6, 1), datetime.date(2023, 7, 19), datetime.date(2024, 11, 28), datetime.date(2024, 12, 3), datetime.date(2025, 1, 16), datetime.date(2025, 2, 5), datetime.date(2025, 3, 3), datetime.date(2025, 4, 9), datetime.date(2025, 6, 15), datetime.date(2025, 7, 19), datetime.date(2025, 8, 5), datetime.date(2025, 11, 11), datetime.date(2026, 1, 3), datetime.date(2026, 2, 3), datetime.date(2026, 3, 3), datetime.date(2026, 4, 18), datetime.date(2026, 6, 10), datetime.date(2026, 7, 8)]
                        min   max  count
plan        duration                    
Monthly HD  1 month     199   199      2
Premium  HD 6 months    599   699      7
Premium 4k  12 months  1199  1499     10
Premium HD  12 months 

In [41]:
zee = pd.read_csv("zee5_in_raw.csv", parse_dates=["date"])
zee["plan"] = zee["plan"].str.replace(r"\s+", " ", regex=True).str.strip()
zee.to_csv("zee5_in_raw.csv", index=False)

print(zee.pivot_table(index="date", columns=["plan", "duration"], values="price"))
print()
print(zee.pivot_table(index="date", columns=["plan", "duration"], values="list_price"))

ts, original = next(r for r in snaps if r[0].startswith("20250116"))
raw = htmllib.unescape(get_html(ts, original))
hits = list(re.finditer(r"₹\s?\d[\d,]*", raw))
print("\n20250116 ->", len(hits), "price mentions")
for m in hits[:6]:
    print(raw[max(0, m.start() - 120): m.start() + 80].replace("\n", " "))
    print("---")

plan       Monthly HD Premium 4k Premium HD         
duration      1 month  12 months  12 months 6 months
date                                                
2023-09-30        NaN     1499.0      899.0      NaN
2023-12-07        NaN     1199.0      899.0    599.0
2024-01-02        NaN     1199.0      899.0    599.0
2024-02-15        NaN     1199.0      999.0    699.0
2024-04-17        NaN     1199.0      899.0    699.0
2024-05-17        NaN     1199.0      899.0    699.0
2024-06-08        NaN     1199.0      899.0    699.0
2024-08-03        NaN     1199.0      899.0    699.0
2024-09-16      199.0     1199.0      899.0      NaN
2024-10-01      199.0     1299.0      999.0      NaN

plan       Monthly HD Premium 4k Premium HD         
duration      1 month  12 months  12 months 6 months
date                                                
2023-09-30        NaN     1999.0      999.0      NaN
2023-12-07        NaN     1999.0      999.0    599.0
2024-01-02        NaN     1999.0      999.0  

In [42]:
def zee_cards_new(html):
    t = htmllib.unescape(html)
    t = re.sub(r"<style.*?</style>", "", t, flags=re.S)
    pat = (r'data-test-id="discounted-price_(\w+)"[^>]*>₹\s?([\d,]+)[^<]*</div>'
           r'\s*<div[^>]*data-test-id="price_\1"[^>]*>₹\s?([\d,]+)[^<]*</div>'
           r'.{0,400}?data-test-id="plan-title_\1"[^>]*>([^<]+)<')
    out = []
    for code, lp, p, title in re.findall(pat, t, flags=re.S):
        out.append((code, title.strip(), int(lp.replace(",", "")), int(p.replace(",", ""))))
    return out

failed = set(zbad)
new_rows, still_bad = [], []
for ts, original in snaps:
    date = datetime.strptime(ts, "%Y%m%d%H%M%S").date()
    if date not in failed:
        continue
    cards = zee_cards_new(get_html(ts, original))
    if cards:
        for code, title, lp, p in cards:
            new_rows.append({"service": "ZEE5", "region": "IN", "date": date,
                             "plan": code, "duration": title, "list_price": lp,
                             "price": p, "source": "wayback"})
    else:
        still_bad.append(date)
    time.sleep(3)

zee = pd.concat([pd.read_csv("zee5_in_raw.csv", parse_dates=["date"]),
                 pd.DataFrame(new_rows)], ignore_index=True)
zee["date"] = pd.to_datetime(zee["date"])
zee = zee.drop_duplicates().sort_values(["date", "plan"]).reset_index(drop=True)
zee.to_csv("zee5_in_raw.csv", index=False)

print(len(zee), "rows from", zee["date"].nunique(), "snapshots")
print(len(still_bad), "snapshots still unreadable:", still_bad)
print(zee[zee["date"] >= "2024-11-01"].pivot_table(
    index="date", columns="plan", values="price"))

45 rows from 16 snapshots
19 snapshots still unreadable: [datetime.date(2019, 10, 8), datetime.date(2022, 9, 14), datetime.date(2022, 10, 4), datetime.date(2022, 11, 11), datetime.date(2023, 1, 30), datetime.date(2023, 3, 6), datetime.date(2023, 4, 1), datetime.date(2023, 6, 1), datetime.date(2023, 7, 19), datetime.date(2025, 6, 15), datetime.date(2025, 7, 19), datetime.date(2025, 8, 5), datetime.date(2025, 11, 11), datetime.date(2026, 1, 3), datetime.date(2026, 2, 3), datetime.date(2026, 3, 3), datetime.date(2026, 4, 18), datetime.date(2026, 6, 10), datetime.date(2026, 7, 8)]
plan        Premium4K  PremiumHD  South4K
date                                     
2024-11-28     1299.0      999.0    749.0
2024-12-03     1299.0      999.0    149.0
2025-01-16      299.0      199.0    149.0
2025-02-05      299.0      199.0    149.0
2025-03-03      299.0        NaN    149.0
2025-04-09      299.0        NaN    149.0


In [43]:
new = zee[zee["date"] >= "2024-11-01"]
print(new.groupby(["plan", "duration"]).size())
print()
print(new[["date", "plan", "duration", "list_price", "price"]].to_string(index=False))

plan       duration                  
Premium4K  Premium  4K- All Languages    2
           Premium 4K                    2
           ZEE5 All Access               2
PremiumHD  Monthly HD                    2
           Premium HD                    2
South4K    South 4K                      6
dtype: int64

      date      plan                   duration  list_price  price
2024-11-28 Premium4K                 Premium 4K        1999   1299
2024-11-28 PremiumHD                 Premium HD        1199    999
2024-11-28   South4K                   South 4K        1199    749
2024-12-03 Premium4K                 Premium 4K        1999   1299
2024-12-03 PremiumHD                 Premium HD        1199    999
2024-12-03   South4K                   South 4K         249    149
2025-01-16 Premium4K Premium  4K- All Languages         399    299
2025-01-16 PremiumHD                 Monthly HD         299    199
2025-01-16   South4K                   South 4K         249    149
2025-02-05 Premium4K

In [44]:
zee = pd.read_csv("zee5_in_raw.csv", parse_dates=["date"])

def tier(row):
    s = (str(row["plan"]) + str(row["duration"])).lower().replace(" ", "")
    if "south" in s:
        return "South 4K"
    if "4k" in s or "allaccess" in s:
        return "Premium 4K"
    return "Premium HD"

def billing(row):
    d = str(row["duration"]).lower()
    if "12 month" in d:
        return "annual"
    if "6 month" in d:
        return "6 months"
    if "1 month" in d:
        return "monthly"
    return "monthly" if row["price"] <= 399 else "annual"

zee["tier"] = zee.apply(tier, axis=1)
zee["billing"] = zee.apply(billing, axis=1)

zclean = (zee.drop_duplicates(["date", "tier", "billing"])
             [["service", "region", "date", "tier", "billing",
               "list_price", "price", "source"]]
             .sort_values(["tier", "billing", "date"])
             .reset_index(drop=True))
zclean.to_csv("zee5_in_clean.csv", index=False)

print(len(zclean), "rows")
print(zclean.groupby(["tier", "billing"])["price"].agg(["min", "max", "count"]))

45 rows
                      min   max  count
tier       billing                    
Premium 4K annual    1199  1499     12
           monthly    299   299      4
Premium HD 6 months   599   699      7
           annual     899   999     12
           monthly    199   199      4
South 4K   annual     749   749      1
           monthly    149   149      5


In [45]:
cols = ["service", "region", "date", "series", "billing",
        "price", "list_price", "source", "flag"]

sp = pd.read_csv("spotify_in_clean.csv", parse_dates=["date"])
sp["billing"] = "monthly"
sp["series"] = "Spotify | " + sp["plan_group"] + " | monthly"
sp["price"] = sp["regular_price"]
sp["list_price"] = pd.NA

yt = pd.read_csv("youtube_in.csv", parse_dates=["date"])
yt["billing"] = "monthly"
yt["series"] = "YouTube | " + yt["plan"] + " | monthly"
yt["price"] = yt["regular_price"]
yt["list_price"] = pd.NA
yt["flag"] = ""

hs = pd.read_csv("jiohotstar_in.csv", parse_dates=["date"])
hs["series"] = "JioHotstar | " + hs["plan"] + " | " + hs["billing"]
hs["price"] = hs["regular_price"]
hs["list_price"] = pd.NA
hs["flag"] = ""

ze = pd.read_csv("zee5_in_clean.csv", parse_dates=["date"])
ze["series"] = "ZEE5 | " + ze["tier"] + " | " + ze["billing"]
ze["flag"] = ""

if "flag" not in sp.columns:
    sp["flag"] = ""
sp["flag"] = sp["flag"].fillna("")

allp = (pd.concat([d[cols] for d in (sp, yt, hs, ze)], ignore_index=True)
          .sort_values(["series", "date"]).reset_index(drop=True))

allp["prev_price"] = allp.groupby("series")["price"].shift()
allp["change_pct"] = ((allp["price"] / allp["prev_price"] - 1) * 100).round(1)
allp.to_csv("all_prices.csv", index=False)

summary = allp.groupby("series").agg(
    rows=("price", "size"),
    first=("date", "min"),
    last=("date", "max"),
    changes=("change_pct", lambda s: int((s.fillna(0) != 0).sum())),
)
print(len(allp), "rows,", len(summary), "series")
print(summary.to_string())

606 rows, 26 series
                                     rows      first       last  changes
series                                                                  
JioHotstar | Mobile | annual           20 2025-02-01 2026-09-01        0
JioHotstar | Mobile | monthly           8 2026-02-01 2026-09-01        0
JioHotstar | Mobile | quarterly        20 2025-02-01 2026-09-01        0
JioHotstar | Premium | annual          20 2025-02-01 2026-09-01        1
JioHotstar | Premium | monthly          8 2026-02-01 2026-09-01        0
JioHotstar | Premium | quarterly       20 2025-02-01 2026-09-01        1
JioHotstar | Super | annual            20 2025-02-01 2026-09-01        1
JioHotstar | Super | monthly            8 2026-02-01 2026-09-01        0
JioHotstar | Super | quarterly         20 2025-02-01 2026-09-01        1
Spotify | Duo | monthly                50 2021-02-24 2025-10-08        3
Spotify | Family | monthly             50 2021-02-24 2025-10-08        3
Spotify | Individual/Lite | mon